# SEOSS Dataset: GPU-Accelerated Change Impact Analysis & Traceability Pipeline

**Target Architecture:** 8GB VRAM Local GPU (RTX 3060/3070/4060) or Google Colab (Tesla T4 / V100 / A100)

**Dataset:** The SEOSS Dataset (*Rath & M\u00e4der*, 33 SQLite project databases, 278,563 issues, 96,750 ground-truth links, 2.84 GB)

**Key Architecture:**
- Direct SQLite ingestion across all 33 projects
- PyTorch CUDA FP16 Autocast for maximum throughput on consumer GPUs
- Chunked GPU Matrix Multiplication (`torch.matmul` + `torch.topk`) to bypass the $O(N^2)$ Cartesian explosion
- Rigorous empirical Software Engineering metrics: Accuracy, Precision, Recall, F1, F2, Recall@K, MAP, MRR, Inspection Effort
- Zero hypothesis testing or synthetic p-values \u2014 pure empirical performance

## Step 1: Verify Colab GPU Hardware
Confirm that CUDA is available and note the allocated GPU model.

In [ ]:
!nvidia-smi

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: No GPU detected! Go to 'Runtime > Change runtime type' and select GPU (T4).")

## Step 2: Install High-Performance Dependencies
Colab GPU environments already include CUDA-enabled PyTorch. We install the required NLP and metric packages.

In [ ]:
!pip install -q -U sentence-transformers transformers scikit-learn pandas numpy matplotlib tqdm pyyaml

## Step 3: Setup Working Directory and Dataset Location
This cell locates the pipeline directory containing `run_full_pipeline.py` and the `dataverse_files` dataset directory.

In [ ]:
import os
import sys

# Optional: Mount Google Drive if your files are stored in Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Google Drive mounting skipped or already active.")

# 1. Automatically locate the pipeline code directory
pipeline_candidates = [
    "/content/seoss_gpu_pipeline",
    "/content/drive/MyDrive/seoss_gpu_pipeline",
    "/content/drive/MyDrive/SEOSS_Full_GPU_Pipeline",
    "/content",
    os.getcwd()
]

PIPELINE_DIR = None
for p in pipeline_candidates:
    if os.path.exists(os.path.join(p, "run_full_pipeline.py")):
        PIPELINE_DIR = p
        break

# If not found, check if a ZIP was uploaded to /content
if PIPELINE_DIR is None:
    for zip_name in ["SEOSS_Full_GPU_Pipeline.zip", "seoss_gpu_pipeline.zip"]:
        if os.path.exists(f"/content/{zip_name}"):
            print(f"Extracting {zip_name} to /content/seoss_gpu_pipeline...")
            !unzip -q "/content/{zip_name}" -d /content/seoss_gpu_pipeline
            PIPELINE_DIR = "/content/seoss_gpu_pipeline"
            break

if PIPELINE_DIR is None:
    raise FileNotFoundError(
        "Could not locate 'run_full_pipeline.py'. Please upload the pipeline folder or "
        "SEOSS_Full_GPU_Pipeline.zip to Colab (/content) and re-run this cell."
    )

print(f"Pipeline codebase active at: {PIPELINE_DIR}")
# Change working directory to the pipeline directory
%cd "$PIPELINE_DIR"

# 2. Automatically locate the dataverse_files folder
data_candidates = [
    "/content/dataverse_files",
    "/content/drive/MyDrive/dataverse_files",
    os.path.join(PIPELINE_DIR, "dataverse_files"),
    os.path.join(PIPELINE_DIR, "..", "dataverse_files"),
    "./dataverse_files"
]

DATA_DIR = None
for d in data_candidates:
    if d and os.path.exists(d):
        sqlite_count = len([f for f in os.listdir(d) if f.endswith('.sqlite3')])
        if sqlite_count > 0:
            DATA_DIR = os.path.abspath(d)
            print(f"Found dataset at: {DATA_DIR} ({sqlite_count} SQLite databases)")
            break

if DATA_DIR is None:
    print("\nWARNING: 'dataverse_files' directory was not found automatically.")
    print("Please set DATA_DIR manually below before running the pipeline cells.")
    DATA_DIR = "/content/dataverse_files"

## Step 4: Run Smoke Test (2 Projects)
Quickly verifies end-to-end data loading, GPU encoding, chunked retrieval, and metric reporting on 2 small projects before running the full dataset.

In [ ]:
!python run_full_pipeline.py \
    --data_dir "$DATA_DIR" \
    --output_dir "./results_smoke_test" \
    --projects "archiva,derby" \
    --device cuda \
    --batch_size 256 \
    --chunk_size 500 \
    --top_k 20 \
    --threshold 0.5

## Step 5: Run Full 3GB Dataset Pipeline (All 33 Projects)
Executes dense encoding, chunked GPU matrix multiplication, and metric aggregation across all 33 projects.

In [ ]:
!python run_full_pipeline.py \
    --data_dir "$DATA_DIR" \
    --output_dir "./results_full_33_projects" \
    --projects all \
    --device cuda \
    --batch_size 256 \
    --chunk_size 500 \
    --top_k 20 \
    --threshold 0.5

## Step 6: View Formatted Benchmark Results & Visualizations

In [ ]:
import os
import pandas as pd
from IPython.display import display, Image

results_dir = './results_full_33_projects'

# Per-Project Metrics Table
per_proj_path = os.path.join(results_dir, 'per_project_metrics.csv')
if os.path.exists(per_proj_path):
    per_proj_df = pd.read_csv(per_proj_path)
    print("PER-PROJECT BENCHMARK METRICS:")
    display(per_proj_df)

# Overall Summary Table (Macro vs Micro Average)
summary_path = os.path.join(results_dir, 'overall_summary.csv')
if os.path.exists(summary_path):
    summary_df = pd.read_csv(summary_path)
    print("\nOVERALL AGGREGATE SUMMARY:")
    display(summary_df)

# Check for any skipped or failed projects
failed_path = os.path.join(results_dir, 'failed_projects.csv')
if os.path.exists(failed_path):
    print("\nFAILED / SKIPPED PROJECTS:")
    display(pd.read_csv(failed_path))
else:
    print("\nAll requested projects processed completely.")

# Display Generated Publication Charts
print("\nRecall@K Performance Curve:")
plot_recall = os.path.join(results_dir, 'plots', 'recall_at_k_curve.png')
if os.path.exists(plot_recall):
    display(Image(plot_recall))

print("\nPer-Project Comparison (F1, MAP, Recall@10):")
plot_comp = os.path.join(results_dir, 'plots', 'per_project_comparison.png')
if os.path.exists(plot_comp):
    display(Image(plot_comp))

print("\nPrecision-Recall Operating Curve:")
plot_pr = os.path.join(results_dir, 'plots', 'precision_recall_operating_curve.png')
if os.path.exists(plot_pr):
    display(Image(plot_pr))

## Step 7: Download Benchmark Package
Zips all CSV metrics, JSON summaries, and generated plots for easy download to your local machine.

In [ ]:
from google.colab import files

!zip -r seoss_benchmark_results.zip ./results_full_33_projects
files.download('seoss_benchmark_results.zip')